# Clase 183 — Bootstrap y permutation tests

Resampling en lugar de supuestos paramétricos: el **bootstrap** estima la distribución muestral de cualquier estadístico; el **permutation test** produce un p-value re-mezclando etiquetas. Usamos las APIs modernas de scipy (≥ 1.9), incluida la variante **BCa**.

Requiere: `numpy`, `scipy`, `scikit-learn`, `matplotlib`.

## 1. Bootstrap a mano vs scipy

`B` resamples con reemplazo del mismo tamaño; se toman los cuantiles 2.5 % y 97.5 %.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

tip = rng.gamma(shape=2.0, scale=1.5, size=200)   # asimétrico positivo
B = 10_000
boot_means = np.array([rng.choice(tip, size=len(tip), replace=True).mean() for _ in range(B)])
lo, hi = np.percentile(boot_means, [2.5, 97.5])
sp = stats.bootstrap((tip,), np.mean, n_resamples=B, method="percentile", random_state=rng)
print(f"manual  IC95% = ({lo:.3f}, {hi:.3f})")
print(f"scipy   IC95% = ({sp.confidence_interval.low:.3f}, {sp.confidence_interval.high:.3f})")
assert abs(lo - sp.confidence_interval.low) < 0.1

plt.figure(figsize=(6, 4))
plt.hist(boot_means, bins=50, color="darkorange", alpha=0.7)
plt.axvline(lo, color="k", ls="--"); plt.axvline(hi, color="k", ls="--")
plt.title("Distribución bootstrap de la media (IC95% percentil)")
plt.tight_layout(); plt.show()

## 2. BCa vs percentil

BCa corrige sesgo y asimetría. Con datos lognormales, el IC de la mediana queda asimétrico hacia la cola derecha (refleja la realidad).

In [ ]:
data = rng.lognormal(0, 1, 60)
perc = stats.bootstrap((data,), np.median, n_resamples=10_000, method="percentile", random_state=rng)
bca  = stats.bootstrap((data,), np.median, n_resamples=10_000, method="BCa", random_state=rng)
med = np.median(data)
print(f"mediana puntual = {med:.3f}")
print(f"percentil: ({perc.confidence_interval.low:.3f}, {perc.confidence_interval.high:.3f})")
print(f"BCa:       ({bca.confidence_interval.low:.3f}, {bca.confidence_interval.high:.3f})")

## 3. IC bootstrap para el AUC de un modelo

Bootstrap sobre los índices del test para poner un IC95 % alrededor del AUC.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)

def auc_stat(idx):
    return roc_auc_score(yte[idx], proba[idx])

idx = np.arange(len(yte))
res = stats.bootstrap((idx,), auc_stat, n_resamples=2000, method="percentile",
                      random_state=rng, vectorized=False)
print(f"AUC test = {auc:.3f}  IC95% = ({res.confidence_interval.low:.3f}, {res.confidence_interval.high:.3f})")
assert res.confidence_interval.low <= auc <= res.confidence_interval.high

## 4. Permutation test bilateral

Bajo `H₀` de "no diferencia", las etiquetas son intercambiables. Re-mezclarlas genera la distribución del estadístico bajo `H₀`.

In [ ]:
a = rng.normal(3.0, 1.0, 100)
b = rng.normal(3.4, 1.0, 110)

def diff_means(x, y):
    return np.mean(x) - np.mean(y)

pt = stats.permutation_test((a, b), diff_means, n_resamples=10_000,
                            alternative="two-sided", random_state=rng)
mw = stats.mannwhitneyu(a, b).pvalue
print(f"permutation p={pt.pvalue:.4f}   Mann-Whitney p={mw:.4f}")
print("Ambos coinciden cualitativamente (hay diferencia real).")
assert pt.pvalue < 0.05

## Ejercicios

1. Comparativa de cobertura: simulá 500 datasets de `Exp(1)` con `n=25` y contá la cobertura empírica del IC de la mediana con `percentile` y con `BCa`.
2. Reportá `p < 1/(n_resamples+1)` cuando el permutation test devuelve el p mínimo posible.
3. Calculá un IC bootstrap para la diferencia de medianas entre dos grupos y relacionalo con el p-value de permutación.

## Conclusiones

- El bootstrap estima la **variabilidad** de cualquier estadístico (media, mediana, AUC, R²) sin fórmula cerrada.
- Usá `B = 10 000` para IC95 %; **BCa** corrige sesgo y asimetría (mejor cobertura que percentil).
- El permutation test da un p-value exacto condicional a los datos, sin supuestos distribucionales.
- El bootstrap asume independencia: para series temporales usá block bootstrap; con `n < 20` preferí métodos paramétricos.